# State-space analysis of the 2-unit GRU — v8

Five figures, all in the 2D hidden-state plane (hidden unit 1 vs hidden unit 2):

1. **All twelve subtasks, averaged** — one mean trajectory per subtask (averaged over its 100 test trials).
   The three conflict subtasks are split by the network's output label, giving two lines each (six conflict lines).
2. **Conflict subtasks on their own** — the same six conflict trajectories, isolated.
3. **Conflict by stimulus strength** — conflict trajectories coloured by a gradient set by the intensity gap
   between the two competing cues.
4. **Variance across seeds** — for a single subtask, the averaged trajectory of each of the five seeds.
5. **Decision boundaries across seeds** — the readout boundaries (lines only, no fill) overlaid for five seeds.

Loads data from `./generated_trials_v8`. Trains five seeds of a 2-unit unified GRU and runs all five
figures on them. Built on the validated hand-coded GRU update (matches PyTorch to ~1e-7).

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8")
OUT_DIR = Path("./statespace_v8"); OUT_DIR.mkdir(exist_ok=True)

HIDDEN = 2
T_ON, T_OFF = 10, 20                      # stimulus window in timesteps
CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]   # grey / orange / red / blue

## 2. Load data and subtask groups

In [ ]:
def load(split):
    d = np.load(DATA_DIR / f"{split}.npz", allow_pickle=True)
    out = {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64), "types": d["types"]}
    if "aud_int" in d:
        out["aud_int"] = d["aud_int"].astype(np.float32); out["vis_int"] = d["vis_int"].astype(np.float32)
    return out

train, test = load("train"), load("test")
T = test["X"].shape[2]
SUBTASKS = sorted(set(test["types"]))
CONFLICT = ["det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
print("Train:", train["X"].shape, " Test:", test["X"].shape)
print("Conflict subtasks:", CONFLICT)

## 3. Model

In [ ]:
class UnifiedGRU(nn.Module):
    def __init__(self, n_channels=4, hidden_size=2, n_classes=4):
        super().__init__()
        self.gru = nn.GRU(n_channels, hidden_size, batch_first=True)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2)
        h, _ = self.gru(x)
        return self.readout(h)

## 4. Train five seeds

We keep all five (not just the best), because figures 4 and 5 compare across seeds. Training is quick
at two units. Each model is stored with its readout weights.

In [ ]:
def train_model(seed, n_epochs=50, lr=1e-3, batch=64):
    torch.manual_seed(seed); np.random.seed(seed)
    model = UnifiedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(train["y"])),
                        batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            logits = model(Xb); B, Tt, C = logits.shape
            loss = loss_fn(logits.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def nonconf_acc(model):
    model.eval()
    with torch.no_grad():
        pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    return np.mean([(pred[test["types"] == s] == test["y"][test["types"] == s]).mean() for s in NONCONF])

SEEDS = [0, 1, 2, 3, 4]
models = {}
for s in SEEDS:
    m = train_model(s); models[s] = m
    print("seed %d  non-conflict acc %.3f" % (s, nonconf_acc(m)))

## 5. Hand-coded GRU update (verified) and per-seed helpers

The trajectories use the reconstructed numpy update so they are consistent with the fixed-point/flow work, and so the averaged trajectories are computed identically across seeds. We re-verify the match here.

In [ ]:
def gru_params(model):
    g = model.gru
    Wih = g.weight_ih_l0.detach().numpy(); Whh = g.weight_hh_l0.detach().numpy()
    bih = g.bias_ih_l0.detach().numpy();   bhh = g.bias_hh_l0.detach().numpy()
    H = Whh.shape[1]; sl = lambda M, i: M[i*H:(i+1)*H]
    return dict(Wir=sl(Wih,0), Wiz=sl(Wih,1), Win=sl(Wih,2),
                Whr=sl(Whh,0), Whz=sl(Whh,1), Whn=sl(Whh,2),
                bir=bih[:H], biz=bih[H:2*H], bin_=bih[2*H:3*H],
                bhr=bhh[:H], bhz=bhh[H:2*H], bhn=bhh[2*H:3*H], H=H)

def sigmoid(v): return 1.0 / (1.0 + np.exp(-v))

def gru_step(h, x, p):
    r = sigmoid(p["Wir"] @ x + p["bir"] + p["Whr"] @ h + p["bhr"])
    z = sigmoid(p["Wiz"] @ x + p["biz"] + p["Whz"] @ h + p["bhz"])
    n = np.tanh(p["Win"] @ x + p["bin_"] + r * (p["Whn"] @ h + p["bhn"]))
    return (1.0 - z) * n + z * h

def readout_of(model):
    return model.readout.weight.detach().numpy(), model.readout.bias.detach().numpy()

def hidden_trajectories(model, idx):
    """Return array (n_trials, T+1, HIDDEN) of hidden states (start at origin) for given test indices."""
    p = gru_params(model)
    out = np.zeros((len(idx), T + 1, HIDDEN))
    for k, i in enumerate(idx):
        h = np.zeros(HIDDEN)
        out[k, 0] = h
        for t in range(T):
            h = gru_step(h, test["X"][i][:, t], p); out[k, t + 1] = h
    return out

def predictions(model, idx):
    model.eval()
    with torch.no_grad():
        pr = model(torch.from_numpy(test["X"][idx]))[:, -1, :].argmax(-1).numpy()
    return pr

# verify on seed 0
_p = gru_params(models[SEEDS[0]])
xb = torch.from_numpy(test["X"][:6])
with torch.no_grad(): h_pt, _ = models[SEEDS[0]].gru(xb.transpose(1, 2))
h_pt = h_pt.numpy(); err = 0.0
for b in range(6):
    h = np.zeros(HIDDEN)
    for t in range(T):
        h = gru_step(h, test["X"][b][:, t], _p); err = max(err, abs(h - h_pt[b, t]).max())
print("max |manual - pytorch| =", err); assert err < 1e-4

## 6. Decision-region backdrop (shared helper)

In [ ]:
GRID = np.linspace(-1.05, 1.05, 400)
GX, GY = np.meshgrid(GRID, GRID)
_flat = np.stack([GX.ravel(), GY.ravel()], 1)

def draw_regions(ax, model, fill=True):
    Wro, bro = readout_of(model)
    region = np.argmax(_flat @ Wro.T + bro, axis=1).reshape(GX.shape)
    if fill:
        ax.pcolormesh(GX, GY, region, cmap=ListedColormap(CLASS_COLORS), alpha=0.20, shading="auto", vmin=0, vmax=3)
    ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2")
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_aspect("equal")
    return region

def mean_trajectory(model, idx):
    return hidden_trajectories(model, idx).mean(0)   # (T+1, HIDDEN)

def conflict_label_groups(subtask):
    """For a conflict subtask, the two output labels we split on."""
    if subtask == "det_multisensory":
        return [("det", 1), ("no-det", 0)]
    return [("right", 2), ("left", 3)]   # localisation conflicts

## Figure 1 — all twelve subtasks, averaged

One mean trajectory per non-conflict subtask. Each conflict subtask is split by the network's output label
into two averaged trajectories, so the conflicts contribute six lines in total.

In [ ]:
DISPLAY_SEED = SEEDS[0]      # which seed figures 1-3 show; set to any value in SEEDS (no quality filtering)
model = models[DISPLAY_SEED]
fig, ax = plt.subplots(figsize=(8.5, 8))
draw_regions(ax, model)

base_cols = plt.cm.tab20(np.linspace(0, 1, 20))
ci = 0
def nextcol():
    global ci; c = base_cols[ci % 20]; ci += 1; return c

# non-conflict subtasks: one averaged line each
for s in NONCONF:
    idx = np.where(test["types"] == s)[0]
    traj = mean_trajectory(model, idx)
    c = nextcol()
    ax.plot(traj[:, 0], traj[:, 1], color=c, lw=2, label=s)
    ax.scatter(traj[-1, 0], traj[-1, 1], color=c, s=45, marker="*", edgecolor="k", lw=0.4, zorder=5)

# conflict subtasks: split by output label -> two averaged lines each
pred_all = predictions(model, np.arange(len(test["X"])))
for s in CONFLICT:
    for name, lab in conflict_label_groups(s):
        idx = np.where((test["types"] == s) & (pred_all == lab))[0]
        if len(idx) == 0: continue
        traj = mean_trajectory(model, idx)
        c = nextcol()
        ax.plot(traj[:, 0], traj[:, 1], color=c, lw=2.4, ls="--",
                label="%s -> %s" % (s, name))
        ax.scatter(traj[-1, 0], traj[-1, 1], color=c, s=60, marker="*", edgecolor="k", lw=0.5, zorder=6)

ax.scatter(0, 0, color="k", s=30, zorder=7)
ax.set_title("Figure 1: average state-space trajectory per subtask, seed %d (conflicts dashed, split by output label)" % DISPLAY_SEED)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=7.5, frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig1_all_subtasks.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 2 — conflict subtasks on their own

The six conflict trajectories isolated, so the split by output label is easy to read.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7.5))
draw_regions(ax, model)
ci = 0
for s in CONFLICT:
    for name, lab in conflict_label_groups(s):
        idx = np.where((test["types"] == s) & (pred_all == lab))[0]
        if len(idx) == 0: continue
        traj = mean_trajectory(model, idx)
        c = nextcol()
        ax.plot(traj[:, 0], traj[:, 1], color=c, lw=2.6, label="%s -> %s (n=%d)" % (s, name, len(idx)))
        ax.scatter(traj[T_OFF, 0], traj[T_OFF, 1], color=c, s=30, zorder=5)         # stimulus offset
        ax.scatter(traj[-1, 0], traj[-1, 1], color=c, s=70, marker="*", edgecolor="k", lw=0.5, zorder=6)
ax.scatter(0, 0, color="k", s=30, zorder=7)
ax.set_title("Figure 2: conflict trajectories, split by output label (seed %d)" % DISPLAY_SEED)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig2_conflicts.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 3 — conflict trajectories by intensity gap (gradient)

Localisation-conflict trajectories coloured by the intensity gap between the two competing cues. Small gap
(near-tie) to large gap (one cue clearly dominant). A systematic shift of the endpoint with the gap is the
reliability-weighting effect made visual.

In [ ]:
LOC_CONFLICTS = ["loc_conflict_audL_visR", "loc_conflict_audR_visL"]
idx = np.where(np.isin(test["types"], LOC_CONFLICTS))[0]
gap = np.abs(test["aud_int"][idx] - test["vis_int"][idx])
trajs = hidden_trajectories(model, idx)               # (n, T+1, 2)

order = np.argsort(gap)
cmap = plt.cm.viridis
gmin, gmax = gap.min(), gap.max()

fig, ax = plt.subplots(figsize=(8.4, 7.5))
draw_regions(ax, model)
for k in order:
    c = cmap((gap[k] - gmin) / (gmax - gmin + 1e-9))
    ax.plot(trajs[k, :, 0], trajs[k, :, 1], color=c, lw=0.8, alpha=0.5)
    ax.scatter(trajs[k, -1, 0], trajs[k, -1, 1], color=c, s=18, zorder=5)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(gmin, gmax)); sm.set_array([])
fig.colorbar(sm, ax=ax, label="intensity gap  |stronger - weaker|", fraction=0.046)
ax.scatter(0, 0, color="k", s=30, zorder=7)
ax.set_title("Figure 3: localisation-conflict trajectories coloured by intensity gap (seed %d)" % DISPLAY_SEED)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig3_gap_gradient.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 4 — variance across seeds (one subtask per plot)

For a chosen subtask, the averaged trajectory of each of the five seeds, overlaid. Tight overlap means the
five networks behave similarly; spread means they diverge. `SUBTASK_FOR_FIG4` selects the subtask.

In [ ]:
SUBTASK_FOR_FIG4 = "loc_conflict_audL_visR"   # any entry of SUBTASKS

# Each seed has its own state space (its own hidden axes / readout), so we draw each seed's own
# faint decision boundary and its averaged trajectory in a matching colour.
seed_cols = plt.cm.tab10(np.linspace(0, 1, len(SEEDS)))
fig, ax = plt.subplots(figsize=(8, 7.5))
for si, s in enumerate(SEEDS):
    m = models[s]
    idx = np.where(test["types"] == SUBTASK_FOR_FIG4)[0]
    traj = mean_trajectory(m, idx)
    ax.plot(traj[:, 0], traj[:, 1], color=seed_cols[si], lw=2, label="seed %d" % s)
    ax.scatter(traj[-1, 0], traj[-1, 1], color=seed_cols[si], s=55, marker="*", edgecolor="k", lw=0.5, zorder=6)
ax.scatter(0, 0, color="k", s=30, zorder=7)
ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2")
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_aspect("equal")
ax.set_title("Figure 4: averaged trajectory across 5 seeds\nsubtask = %s" % SUBTASK_FOR_FIG4)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9, frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR / ("fig4_seeds_%s.png" % SUBTASK_FOR_FIG4), dpi=150, bbox_inches="tight"); plt.show()

print("Note: hidden axes are only defined up to the network's own parametrisation, so seeds need not align")
print("exactly even when they behave equivalently. Figure 5 compares the decision geometry directly.")

## Figure 5 — decision boundaries across seeds

The readout decision regions for all five seeds, drawn as boundary lines only (no fill), overlaid in
different colours. This compares how each network carves the state space into the four classes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7.5))
for si, s in enumerate(SEEDS):
    Wro, bro = readout_of(models[s])
    region = np.argmax(_flat @ Wro.T + bro, axis=1).reshape(GX.shape).astype(float)
    # draw boundaries between differing argmax regions as contour lines
    ax.contour(GX, GY, region, levels=np.arange(0.5, 3.5, 1.0),
               colors=[seed_cols[si]], linewidths=1.6)
    ax.plot([], [], color=seed_cols[si], lw=1.6, label="seed %d" % s)   # legend proxy
ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2")
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_aspect("equal")
ax.set_title("Figure 5: readout decision boundaries across 5 seeds (lines only)")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9, frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig5_boundaries_seeds.png", dpi=150, bbox_inches="tight"); plt.show()

## Notes

- Figures 1 to 3 display `DISPLAY_SEED` (default `SEEDS[0]`); figure 4 overlays all five seeds for one
  subtask and figure 5 uses all five.
- All figures are saved to `./statespace_v8/` as PNGs.
- Caveat for figures 4 and 5: the two hidden axes are internal to each network, so different seeds can
  represent the same computation in rotated or reflected coordinates. Tight agreement is strong evidence of
  shared structure; apparent disagreement may be a change of coordinates rather than a real difference.
  This is the point Maheswaranathan et al. (2019) make, and the reason for comparing at the level of
  dynamical structure rather than raw axes beyond two units.

In [ ]:
print("Saved figures:")
for f in sorted(OUT_DIR.glob("*.png")):
    print("  ", f)